In [1]:
import os
import torch
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
print("Loading Whisper Features...")
path_whisper = "/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/SVM_MEAN_MAX_STD/"
X_train = np.load(path_whisper + "X_train.npy")
y_train = np.load(path_whisper + "y_train.npy") 
X_val = np.load(path_whisper + "X_val.npy")
y_val = np.load(path_whisper + "y_val.npy")
X_test = np.load(path_whisper + "X_test.npy")
y_test = np.load(path_whisper + "y_test.npy")

Loading Whisper Features...


In [3]:
label_map = {
    "eng": 0,
    "hin": 1
}


orig_train_size = len(y_train)
orig_val_size = len(y_val)
orig_test_size = len(y_test)


y_train = np.array([label_map.get(i, -1) for i in y_train])
y_val = np.array([label_map.get(i, -1) for i in y_val])
y_test = np.array([label_map.get(i, -1) for i in y_test])


mask_train = y_train != -1
X_train = X_train[mask_train]
y_train = y_train[mask_train]

mask_val = y_val != -1
X_val = X_val[mask_val]
y_val = y_val[mask_val]

mask_test = y_test != -1
X_test = X_test[mask_test]
y_test = y_test[mask_test]


print("--- Invalid Labels Removed ---")
print(f"Train: {orig_train_size - len(y_train)} samples removed (out of {orig_train_size})")
print(f"Val:   {orig_val_size - len(y_val)} samples removed (out of {orig_val_size})")
print(f"Test:  {orig_test_size - len(y_test)} samples removed (out of {orig_test_size})")
print("------------------------------")

--- Invalid Labels Removed ---
Train: 0 samples removed (out of 35272)
Val:   0 samples removed (out of 5108)
Test:  0 samples removed (out of 9971)
------------------------------


In [4]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [6]:
import joblib


model_path = '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/SVM_MEAN_MAX_STD/svm_model_saved.pkl'

svm = joblib.load(model_path)

print("Model loaded successfully")


Model loaded successfully


In [7]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

/media/nit/DATADRIVE1/AJAY/MLproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-02 17:44:59.972876: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 17:45:03.646026: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-02 17:45:27.027761: I tensorflow/co

In [8]:
def extract_embedding(audio):
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device).to(torch_dtype)
    
    with torch.no_grad():
        encoder_outputs = model.get_encoder()(input_features)
        
    hidden_states = encoder_outputs.last_hidden_state
    
    emb_mean = hidden_states.mean(dim=1).squeeze().cpu().numpy()
    emb_std = hidden_states.std(dim=1).squeeze().cpu().numpy()
    emb_max = hidden_states.max(dim=1)[0].squeeze().cpu().numpy()
    emb = np.hstack((emb_mean, emb_std, emb_max))
    
    return emb

In [13]:
import os
import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm

def extract_svm_probabilities(csv_path, audio_path, output_csv_path, model, scaler):
    df = pd.read_csv(csv_path)
    hin_probs = []
    
    print(f"Extracting SVM Probabilities for {csv_path}...")
    for _, row in tqdm(df.iterrows(), total=len(df)):
        file = os.path.join(audio_path, row['segment_id'])
        
        if not os.path.exists(file):
            hin_probs.append(np.nan) 
            continue
            
        try:
            audio, sr = librosa.load(file, sr=16000)
            start = int(row['start_time'] * sr)
            end = int(row['end_time'] * sr)
            audio_segment = audio[start:end]
            
            if len(audio_segment) < 300:
                hin_probs.append(np.nan)
                continue
            
            try:
                emb = extract_embedding(audio_segment) 
            except RuntimeError:
                pad_frames = 80
                padded_start = max(0, start - pad_frames)
                padded_end = min(len(audio), end + pad_frames)
                padded_audio_segment = audio[padded_start:padded_end]
                emb = extract_embedding(padded_audio_segment)

            emb_scaled = scaler.transform(emb.reshape(1, -1))
            
            probs = model.predict_proba(emb_scaled)[0]
            hin_probs.append(probs[1]) 
            
        except Exception:
            hin_probs.append(np.nan)

    output_df = df[['segment_id', 'start_time', 'end_time', 'label']].copy()
    output_df['whisper_prob_hin'] = hin_probs
    
    output_df = output_df.dropna()
    
    file_exists = os.path.isfile(output_csv_path)
    output_df.to_csv(output_csv_path, mode='a', index=False, header=not file_exists)
    print(f"Saved/Appended to: {output_csv_path}\n")

os.makedirs('/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/', exist_ok=True)

extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_val.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/val_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_val_probs.csv',
    model=svm, 
    scaler=scaler
)
extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_val.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/val_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_val_probs.csv',
    model=svm, 
    scaler=scaler
)

extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_test.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/test_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv',
    model=svm, 
    scaler=scaler
)
extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_test.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/test_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv',
    model=svm, 
    scaler=scaler
)

Extracting SVM Probabilities for /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_val.csv...


100%|███████████████████████████████████████| 3364/3364 [05:33<00:00, 10.10it/s]


Saved/Appended to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_val_probs.csv

Extracting SVM Probabilities for /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_val.csv...


100%|███████████████████████████████████████| 1745/1745 [02:55<00:00,  9.97it/s]


Saved/Appended to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_val_probs.csv

Extracting SVM Probabilities for /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_test.csv...


100%|███████████████████████████████████████| 6546/6546 [10:54<00:00, 10.01it/s]


Saved/Appended to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv

Extracting SVM Probabilities for /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_test.csv...


100%|███████████████████████████████████████| 3433/3433 [05:45<00:00,  9.94it/s]


Saved/Appended to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv



In [ ]:
extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_test.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/test_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv',
    model=svm, 
    scaler=scaler
)
extract_svm_probabilities(
    csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_test.csv',
    audio_path='/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/test_split',
    output_csv_path='/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/whisper_test_probs.csv',
    model=svm, 
    scaler=scaler
)